# fewshot-msme-inspect — Colab / Kaggle setup

Mirrors the local CPU environment so a free GPU burst (Colab T4, Kaggle P100)
can run the heavy bits — feature bank builds, adapter training sweeps — while
every reported latency / RAM number still comes from the local CPU benchmark.

Run the cells top to bottom. Runtime -> Change runtime type -> GPU first.


## 1. What we landed on

In [ ]:
import platform, subprocess, sys

print("python", platform.python_version())
try:
    print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,driver_version",
                          "--format=csv,noheader"], capture_output=True, text=True).stdout.strip()
          or "no GPU reported")
except FileNotFoundError:
    print("no nvidia-smi: CPU runtime")


## 2. Mount Drive (Colab only)

Checkpoints and feature banks survive a runtime reset if they live on Drive.
Skip this cell on Kaggle — use `/kaggle/working` instead.


In [ ]:
IN_COLAB = "google.colab" in sys.modules
DRIVE_ROOT = None

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE_ROOT = "/content/drive/MyDrive/fewshot-msme-inspect"
    import os
    os.makedirs(DRIVE_ROOT, exist_ok=True)
    print("drive root:", DRIVE_ROOT)
else:
    print("not Colab — using local working dir")


## 3. Get the repo

Public repo, so a plain clone is enough. Re-running the cell pulls instead of
re-cloning.


In [ ]:
import os

REPO_URL = "https://github.com/Mounika-Reddy-0802/fewshot-msme-inspect.git"
REPO_DIR = "/content/fewshot-msme-inspect" if IN_COLAB else "/kaggle/working/fewshot-msme-inspect"

if os.path.isdir(os.path.join(REPO_DIR, ".git")):
    !cd {REPO_DIR} && git pull --ff-only
else:
    !git clone {REPO_URL} {REPO_DIR}

os.chdir(REPO_DIR)
sys.path.insert(0, REPO_DIR)
print("cwd:", os.getcwd())


## 4. Install

Colab ships torch already, and it is CUDA-built — leave it alone. We only add
what is missing, so the GPU wheel is not replaced by the CPU one from
`requirements.txt`.


In [ ]:
import torch
print("preinstalled torch:", torch.__version__, "cuda:", torch.cuda.is_available())

!pip install -q open_clip_torch==3.3.0 anomalib==2.6.0 fvcore thop


In [ ]:
# sanity: nothing above downgraded the CUDA build
import open_clip, torch

print("torch    ", torch.__version__, "cuda available:", torch.cuda.is_available())
print("open_clip", open_clip.__version__)
assert torch.cuda.is_available() or not IN_COLAB, "GPU runtime expected: Runtime > Change runtime type"


## 5. Point the roots at persistent storage

Same environment variables as the local setup, so no config or script changes
between machines.


In [ ]:
BASE = DRIVE_ROOT if DRIVE_ROOT else os.getcwd()
os.environ["FEWSHOT_DATA_ROOT"] = os.path.join(BASE, "data")
os.environ["FEWSHOT_RESULTS_ROOT"] = os.path.join(BASE, "results")
os.makedirs(os.environ["FEWSHOT_DATA_ROOT"], exist_ok=True)
os.makedirs(os.environ["FEWSHOT_RESULTS_ROOT"], exist_ok=True)

from src.utils.paths import data_root, results_root
print("data   ", data_root())
print("results", results_root())


## 6. Configs resolve here too

In [ ]:
from src.utils.config import load_config
from src.utils.seed import seed_everything

config = load_config("prototype.yaml")
seed_everything(config["experiment"]["seed"])
print("experiment:", config["experiment"])
print("backbone:  ", config["backbone"])


## 7. Smoke test: encode with the frozen backbone

Downloads the ViT-B/16 weights once (~600 MB) and runs one image and one text
batch through them. If this prints two feature shapes and a similarity, the
environment matches the local one.


In [ ]:
import open_clip, torch

device = "cuda" if torch.cuda.is_available() else "cpu"
name, pretrained = config["backbone"]["name"], config["backbone"]["pretrained"]

model, _, preprocess = open_clip.create_model_and_transforms(name, pretrained=pretrained)
model = model.to(device).eval()
tokenizer = open_clip.get_tokenizer(name)

images = torch.zeros(2, 3, 224, 224, device=device)
text = tokenizer(["a photo of a bottle", "a photo of a bottle with a defect"]).to(device)

with torch.no_grad():
    image_features = model.encode_image(images)
    text_features = model.encode_text(text)
    image_features /= image_features.norm(dim=-1, keepdim=True)
    text_features /= text_features.norm(dim=-1, keepdim=True)
    similarity = image_features @ text_features.T

print("device       ", device)
print("image feats  ", tuple(image_features.shape))
print("text feats   ", tuple(text_features.shape))
print("similarity   ", similarity.cpu().numpy().round(4))


## 8. Record what this runtime was

Colab hands out different GPUs run to run. Anything trained here gets stamped
with the runtime it came from, so results stay traceable.


In [ ]:
import json, subprocess

runtime = {
    "python": platform.python_version(),
    "torch": torch.__version__,
    "device": device,
    "gpu": subprocess.run(["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"],
                          capture_output=True, text=True).stdout.strip() or "cpu",
    "open_clip": open_clip.__version__,
    "backbone": f"{name}/{pretrained}",
}
print(json.dumps(runtime, indent=2))

with open(os.path.join(os.environ["FEWSHOT_RESULTS_ROOT"], "colab_runtime.json"), "w") as f:
    json.dump(runtime, f, indent=2)


---

**Reminder.** Latency, RAM and FLOPs for the paper are measured locally on CPU.
The GPU here is only for building feature banks and training adapter sweeps
faster — never for a number that ends up in the efficiency benchmark.
